# Dear Canada — Place Extraction (pipeline 1 of 2)

Finds place names in every diary entry and attaches each one to its book context: book, chapter, entry, date and the sentence it occurs in. Names that are almost certainly people are filtered out, and every mention gets a suggested **place type** (settlement, region, country, water, landform, facility, vessel) from the words around it. This is what separates *Quebec* the city from *Quebec* the province.

This notebook does **not** decide where a place is. `02_place_matching.ipynb` reads this notebook's output and attaches each name to a geography.

Status: first version of the split pipeline. Extraction uses a pre-trained spaCy model plus rules. The structure is ready for a model fine-tuned on hand-labelled Dear Canada entries to replace the pre-trained one (section 8).

## 0. Requirements, inputs and outputs

### Environment
- Python 3.10 or later, run locally in Jupyter.
- Packages: `pandas`, `spacy` (3.7 or later); `tqdm` optional (progress bars).
- spaCy model: `python -m spacy download en_core_web_trf`. The transformer model is much better than `en_core_web_sm` at telling people from places; it is slower (the first full run takes a while on a CPU), so its output is cached (section 4).

### Network access
None.

### Data the notebook reads
| Item | Location | Format |
|---|---|---|
| Entry files | `ENTRIES_DIR` | One file per book, `<bookcode>_entries.csv`, UTF-8, columns `book, chapter_id, entry_id, date, entry_text` |
| Place lexicon | `resources/place_lexicon.csv` | Hand-edited. Known names with their usual type, optionally by period. Used for typing and to catch names the model misses |
| Not-places list | `resources/not_places.csv` | Hand-edited. Words the model often mistakes for places (kinship terms, "Diary") |
| Gold annotations | `annotation/gold/*.jsonl` | Optional. Hand-labelled entries used for evaluation and training (section 8) |

### Outputs
All outputs are intermediate data and stay inside this pipeline folder (`Pipelines/Place Mentions/`).

| File | Content |
|---|---|
| `intermediate/place_mentions_extracted.csv` | Main result. One row per candidate mention, **including filtered ones** (`keep = False`) so filtering can be reviewed and evaluated. Read by `02_place_matching.ipynb` |
| `intermediate/place_names_by_book.csv` | One row per distinct name per book: counts, label votes, kept/filtered, types. For review |
| `intermediate/ner_cache/<model>/<book>.jsonl` | Raw model output per entry, keyed by a hash of the entry text. Delete to force a re-run |
| `annotation/to_annotate/*.jsonl` | Sample of entries pre-annotated with the current predictions, for hand correction (section 8) |
| `annotation/spacy/*.spacy` | Training and development sets built from the gold annotations (section 8) |

Schema of `place_mentions_extracted.csv`:

| Column | Description |
|---|---|
| `mention_id` | `<book>:<entry_id>:<char_start>`; unique |
| `book`, `chapter_id`, `entry_id` | Position in the corpus |
| `date`, `date_precision`, `year_inferred`, `date_raw` | Entry date in ISO 8601 at day, month or year precision; whether the year came from `DEFAULT_YEARS`; the source string |
| `year` | Year used for period-dependent rules (from `date`, else `DEFAULT_YEARS`) |
| `place` | Cleaned name (leading article, surrounding punctuation and a possessive `'s` removed) |
| `place_raw` | Exact text of the span |
| `place_key` | Folded form of `place` (case, accents, punctuation and Saint/St. removed); the join key used downstream |
| `char_start`, `char_end` | Offsets of `place_raw` in `entry_text` |
| `sentence` | The sentence containing the mention |
| `ner_label` | Model label (`GPE`, `LOC`, `FAC`, `ORG`, or a fine-tuned label); empty for lexicon-only finds |
| `source` | `ner`, `lexicon` (found by the lexicon only) or `ner+lexicon` (lexicon span replaced a shorter model span) |
| `votes` | Place, person and organisation votes this name received across the whole book (section 5.3), e.g. `place:12\|person:1\|org:0` |
| `keep` | True if the mention passed every filter |
| `filter_reason` | Why it was filtered (section 5.3); empty if kept |
| `place_type` | `country`, `region`, `settlement`, `water`, `landform`, `facility`, `vessel` or `unknown` |
| `type_confidence` | `model`, `high`, `medium` or `low` (section 6) |
| `type_evidence` | The rule that set the type, e.g. `name contains 'river'`, `cue 'city of'`, `lexicon` |
| `model` | Model that produced the spans |

### Known limitations
- Story locations described rather than named ("the Halls' farm", "the sheds") are not detected.
- The person filter works per book. A name that is mostly a person in one book and a place in another is judged separately in each, which is intended. A place that shares its name with a main character in the same book (e.g. a girl called Victoria in a book set in Victoria) will be filtered; add it to the lexicon to keep it.
- Type rules look only at the name and a few words around it.

### Paths

In [1]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Every file system location used by the notebook is defined here.
from pathlib import Path

def find_project_root(start=None):
    """The 'Dear Canada' folder: the first parent that contains data/Book Repository."""
    start = Path(start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / "data" / "Book Repository").is_dir():
            return p
    raise FileNotFoundError(f"No parent of {start} contains data/Book Repository")

PROJECT_ROOT  = find_project_root()
PIPELINE_DIR  = PROJECT_ROOT / "Pipelines" / "Place Mentions"

# Inputs
ENTRIES_DIR   = PROJECT_ROOT / "data" / "Book Repository" / "Entries as Csv"
ENTRIES_GLOB  = "*_entries.csv"            # excludes SofS_entries_halved.csv
RESOURCES_DIR = PIPELINE_DIR / "resources"
LEXICON_CSV   = RESOURCES_DIR / "place_lexicon.csv"
NOT_PLACES_CSV = RESOURCES_DIR / "not_places.csv"

# Outputs (intermediate data, kept in this pipeline folder)
INTERMEDIATE_DIR  = PIPELINE_DIR / "intermediate"
NER_CACHE_DIR     = INTERMEDIATE_DIR / "ner_cache"
EXTRACTED_CSV     = INTERMEDIATE_DIR / "place_mentions_extracted.csv"
NAMES_SUMMARY_CSV = INTERMEDIATE_DIR / "place_names_by_book.csv"

# Annotation and training (section 8)
ANNOTATION_DIR  = PIPELINE_DIR / "annotation"
TO_ANNOTATE_DIR = ANNOTATION_DIR / "to_annotate"
GOLD_DIR        = ANNOTATION_DIR / "gold"
SPACY_DATA_DIR  = ANNOTATION_DIR / "spacy"
MODELS_DIR      = ANNOTATION_DIR / "models"

for d in (INTERMEDIATE_DIR, NER_CACHE_DIR, TO_ANNOTATE_DIR, GOLD_DIR):
    d.mkdir(parents=True, exist_ok=True)

for label, p in [("Entries", ENTRIES_DIR), ("Lexicon", LEXICON_CSV), ("Not-places", NOT_PLACES_CSV)]:
    print(f"{label:11}: {p}  {'(exists)' if p.exists() else '(NOT FOUND)'}")
print(f"{'Output':11}: {INTERMEDIATE_DIR}")

Entries    : /Users/sophiebird/Downloads/Dear Canada/data/Book Repository/Entries as Csv  (exists)
Lexicon    : /Users/sophiebird/Downloads/Dear Canada/Pipelines/Place Mentions/resources/place_lexicon.csv  (exists)
Not-places : /Users/sophiebird/Downloads/Dear Canada/Pipelines/Place Mentions/resources/not_places.csv  (exists)
Output     : /Users/sophiebird/Downloads/Dear Canada/Pipelines/Place Mentions/intermediate


### Configuration

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
BOOKS = None                      # None = all books; or a list of codes, e.g. ["SofS", "AToBD"]

# NER model: a spaCy package name, or the path of a fine-tuned model (section 8),
# e.g. MODELS_DIR / "model-best"
SPACY_MODEL    = "en_core_web_trf"
NER_BATCH_SIZE = 8

# How each model label is used.
#   role "place":  the span is a candidate place
#   role "org":    the span is a candidate only if promoted (PROMOTE_ORG)
#   role "person": evidence that a name is a person (used by the person filter)
# The second value is the place type implied by the label, or None to let the rules in section 6 decide.
# Labels not listed are ignored.
MODEL_LABELS = {
    "GPE": ("place", None), "LOC": ("place", None), "FAC": ("place", None),
    "ORG": ("org", None), "PERSON": ("person", None),
    # Labels of a fine-tuned model (section 8). Harmless while the pre-trained model is used.
    "SETTLEMENT": ("place", "settlement"), "REGION": ("place", "region"), "COUNTRY": ("place", "country"),
    "WATER": ("place", "water"), "LANDFORM": ("place", "landform"), "FACILITY": ("place", "facility"),
    "VESSEL": ("place", "vessel"),
}
# Type given when no rule applies, by model label (confidence "low")
LABEL_FALLBACK_TYPE = {"GPE": "settlement", "LOC": "unknown", "FAC": "facility", "ORG": "unknown"}

# ORG spans are kept only if the same name is tagged as a place more often than as a person or
# organisation in the same book, or the name is in the lexicon
PROMOTE_ORG = True

# Titles and kinship words: a capitalised word directly after one of these is a person
PERSON_TITLES = [
    "Mr", "Mrs", "Miss", "Ms", "Master", "Madame", "Mme", "Mademoiselle", "Mlle", "Monsieur", "Dr", "Doctor",
    "Uncle", "Aunt", "Auntie", "Tante", "Oncle", "Cousin", "Grandma", "Grandpa", "Granny", "Grandda", "Granda",
    "Father", "Mother", "Sister", "Brother", "Mère", "Père", "Soeur", "Frère", "Reverend", "Rev", "Pastor",
    "Captain", "Capt", "Sergeant", "Sgt", "Corporal", "Private", "Pte", "Lieutenant", "Lt", "Major", "Colonel",
    "General", "Sir", "Lady", "Lord", "Queen", "King", "Prince", "Princess", "Chief", "Constable", "Nurse",
    "Baba", "Dido", "Moushoom", "Kokum",
]
# Titles that also begin many place names ("Prince Albert", "Queen Charlotte Islands", "Lady Evelyn Lake"):
# a name directly after them is still a person, but a name that starts with them is not filtered
TITLES_ALSO_IN_PLACE_NAMES = {"Queen", "King", "Prince", "Princess", "Lord", "Lady", "Sir", "General", "Chief",
                              "Major", "Captain", "Capt", "Sister", "Brother"}

# Known year per book, used when an entry date has no year (same table as the other pipelines)
DEFAULT_YEARS = {
    "AinUTL": 1666, "AToBD": 1862, "BFfH": 1916, "BUoL": 1885,
    "CofOO": 1866, "DofmyC": 1759, "DofTT": 1887, "DRtoF": 1863,
    "EftW": 1940, "FitS": 1815, "NbutC": 1783, "NSH": 1917,
    "OA": 1922, "OatmyD": 1897, "PasWide": 1926, "PintPL": 1914,
    "SofS": 1847, "TA": 1941, "TFN": 1912, "WofW": 1812,
}

### Type rules

Section 6 applies these in order; the first that matches sets the type. Words are compared after folding (lower case, no accents). Edit the lists as the review tables reveal gaps.

In [3]:
# ── Type rules ────────────────────────────────────────────────────────────────
# 1. A generic word that is part of the name: last word ("Red River") or first word ("Lake Ontario")
TYPE_HEADS_LAST = {
    "water":      ["river", "rivers", "lake", "lakes", "sea", "ocean", "bay", "falls", "harbour", "harbor", "strait",
                   "straits", "sound", "gulf", "creek", "brook", "channel", "rapids", "basin", "cove", "pond", "inlet",
                   "narrows", "canal"],
    "landform":   ["island", "islands", "isle", "mountain", "mountains", "mount", "hill", "hills", "valley", "cape",
                   "point", "peninsula", "plains", "prairie", "prairies", "forest", "woods", "glacier", "pass",
                   "ridge", "rockies"],
    "settlement": ["city", "town", "village", "ville"],
    "region":     ["province", "county", "colony", "territory", "territories", "district", "region", "township",
                   "parish", "settlement"],
    "facility":   ["fort", "church", "cathedral", "chapel", "school", "academy", "college", "university", "hospital",
                   "station", "depot", "street", "avenue", "road", "square", "park", "hall", "house", "hotel", "inn",
                   "mill", "mills", "farm", "camp", "barracks", "prison", "jail", "gaol", "asylum", "wharf", "pier",
                   "dock", "docks", "market", "citadel", "convent", "mission", "post", "cemetery", "bridge",
                   "building", "theatre", "factory", "office", "shed", "sheds", "orphanage", "home"],
}
TYPE_HEADS_FIRST = {
    "water":      ["lake", "lac", "riviere", "river", "baie", "gulf", "bay"],
    "landform":   ["mount", "mt", "cape", "cap", "isle", "ile", "point", "pointe", "grand"],
    "settlement": ["port", "city"],
    "region":     ["province", "county", "colony"],
    "facility":   ["fort", "camp", "pier", "hotel", "st", "ste"],
}
# "grand", "st" and "ste" are too weak to type a name on their own; they are listed so the rule can be
# reviewed, and removed by WEAK_FIRST_WORDS below.
WEAK_FIRST_WORDS = {"grand", "st", "ste"}

# 2. Generic word directly after the name ("Quebec City", "the Ottawa River" when the model tagged only "Ottawa")
#    uses TYPE_HEADS_LAST.

# 3. Cue phrase directly before the name (an optional "the" between cue and name is allowed)
TYPE_CUES_BEFORE = {
    "settlement": ["city of", "town of", "village of", "hamlet of", "port of", "the city", "the town", "the village"],
    "region":     ["province of", "colony of", "county of", "territory of", "district of", "state of", "region of",
                   "parish of", "township of", "dominion of"],
    "country":    ["country of", "kingdom of", "republic of", "empire of"],
    "water":      ["river", "the river", "lake", "the lake", "gulf of", "bay of", "strait of", "straits of",
                   "sea of", "mouth of the", "banks of the", "shore of the", "shores of the"],
    "landform":   ["island of", "isle of", "mount", "cape", "valley of"],
    "vessel":     ["the ship", "ship", "aboard the", "aboard", "on board the", "on board", "the steamer",
                   "steamship", "the steamship", "the boat", "the vessel", "the liner", "ss", "s.s.", "hms",
                   "h.m.s.", "the schooner", "the brig", "the barque", "the ferry"],
}

In [4]:
# ── Imports and dependency check ──────────────────────────────────────────────
import re, json, bisect, hashlib, random, unicodedata
from collections import Counter, defaultdict
from datetime import datetime

import pandas as pd
import spacy

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kwargs):
        return it

print("pandas", pd.__version__, "| spaCy", spacy.__version__, "| model", SPACY_MODEL)

pandas 3.0.0 | spaCy 3.8.16 | model en_core_web_trf


## 1. Load entries

Reads every file matching `ENTRIES_GLOB` in `ENTRIES_DIR` and concatenates them. Required columns are checked per file. Where the `book` column is empty, the book code is taken from the file name.

In [5]:
REQUIRED_COLUMNS = ["book", "chapter_id", "entry_id", "date", "entry_text"]

def load_entries(entries_dir, pattern, books=None):
    files = sorted(Path(entries_dir).glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files matching {pattern} in {entries_dir}")
    frames = []
    for f in files:
        code_from_name = f.name[: -len("_entries.csv")]
        if books is not None and code_from_name not in books:
            continue
        df = pd.read_csv(f, dtype=str, keep_default_na=False, encoding="utf-8")
        missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
        if missing:
            raise ValueError(f"{f.name}: missing columns {missing}")
        df["book"] = df["book"].replace("", code_from_name)
        mismatched = set(df["book"]) - {code_from_name}
        if mismatched:
            print(f"NOTE {f.name}: 'book' column contains {mismatched}, differs from filename code")
        frames.append(df[REQUIRED_COLUMNS])
    if not frames:
        raise ValueError("No books selected; check BOOKS in the configuration cell.")
    return pd.concat(frames, ignore_index=True)

entries = load_entries(ENTRIES_DIR, ENTRIES_GLOB, BOOKS).rename(columns={"date": "date_raw"})

dupes = entries.duplicated(["book", "entry_id"], keep=False).sum()
if dupes:
    print(f"WARNING: {dupes} rows share a (book, entry_id) with another row.")

print(f"{entries['book'].nunique()} books, {len(entries)} entries")
entries.groupby("book").size().rename("entries").to_frame().T

20 books, 3266 entries


book,AToBD,AinUTL,BFfH,BUoL,CofOO,DRtoF,DofTT,DofmyC,EftW,FitS,NSH,NbutC,OA,OatmyD,PasWide,PintPL,SofS,TA,TFN,WofW
entries,114,167,134,129,258,201,169,175,259,142,126,155,123,71,193,230,137,274,70,139


## 2. Date standardisation

`normalise_date` is the existing normalisation routine. Its output (`Month DD, YYYY`) is converted to ISO 8601 with a precision flag by `to_iso`. Partial dates stay partial (`1862-05`, `1862`). `year_inferred` marks dates whose year came from `DEFAULT_YEARS`.

In [6]:
# ── Date normalisation (existing routine) ─────────────────────────────────────
FR_MONTHS = {
    "janvier": "January", "février": "February", "fevrier": "February",
    "mars": "March", "avril": "April", "mai": "May", "juin": "June",
    "juillet": "July", "août": "August", "aout": "August",
    "septembre": "September", "octobre": "October",
    "novembre": "November", "décembre": "December", "decembre": "December"
}
EN_MONTHS = [
    "January","February","March","April","May","June",
    "July","August","September","October","November","December"
]

def normalise_date(raw: str, default_year: int | None = None) -> str:
    if not isinstance(raw, str):
        return "Unknown Date"
    s = raw.strip()

    fr = re.search(r"le\s+(\d{1,2})\s+(" + "|".join(FR_MONTHS) + r")\s+(\d{4})", s, re.I)
    if fr:
        return f"{FR_MONTHS[fr.group(2).lower()]} {fr.group(1).zfill(2)}, {fr.group(3)}"

    dmy = re.search(r"(\d{1,2})\s+(" + "|".join(EN_MONTHS) + r")\s+(\d{4})", s, re.I)
    if dmy:
        return f"{dmy.group(2).capitalize()} {dmy.group(1).zfill(2)}, {dmy.group(3)}"

    mdy = re.search(r"(" + "|".join(EN_MONTHS) + r")\s+(\d{1,2})(?:st|nd|rd|th)?[,\s]+(\d{4})", s, re.I)
    if mdy:
        return f"{mdy.group(1).capitalize()} {mdy.group(2).zfill(2)}, {mdy.group(3)}"

    wd = re.search(r"(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun)[a-z]*,?\s*(" + "|".join(EN_MONTHS) + r")\s+(\d{1,2})", s, re.I)
    if wd:
        m, d = wd.group(1).capitalize(), wd.group(2).zfill(2)
        return f"{m} {d}, {default_year}" if default_year else f"{m} {d}"

    md = re.search(r"(" + "|".join(EN_MONTHS) + r")\s+(\d{1,2})\b", s, re.I)
    if md:
        m, d = md.group(1).capitalize(), md.group(2).zfill(2)
        return f"{m} {d}, {default_year}" if default_year else f"{m} {d}"

    my = re.search(r"(" + "|".join(EN_MONTHS) + r")\s+(\d{4})", s, re.I)
    if my:
        return f"{my.group(1).capitalize()} {my.group(2)}"

    yr = re.search(r"\b(1[5-9]\d{2})\b", s)
    if yr:
        return yr.group(1)

    return s

In [7]:
def to_iso(normalised: str, default_year: int | None):
    """Convert normalise_date output to (iso_string, precision)."""
    s = str(normalised).strip()
    for fmt, precision, out_fmt in (("%B %d, %Y", "day", "%Y-%m-%d"),
                                    ("%B %Y",     "month", "%Y-%m")):
        try:
            return datetime.strptime(s, fmt).strftime(out_fmt), precision
        except ValueError:
            pass
    if re.fullmatch(r"\d{4}", s):
        return s, "year"
    m = re.search(r"\b(1[5-9]\d{2})\b", s)
    if m:
        return m.group(1), "year"
    if default_year:
        return str(default_year), "year"
    return None, "unknown"

entries["default_year"] = entries["book"].map(DEFAULT_YEARS)

no_default = sorted(entries.loc[entries["default_year"].isna(), "book"].unique())
if no_default:
    print(f"NOTE: no DEFAULT_YEARS entry for {len(no_default)} book(s): {no_default}")

def _standardise(row):
    dy = None if pd.isna(row["default_year"]) else int(row["default_year"])
    normalised = normalise_date(row["date_raw"], dy)
    iso, precision = to_iso(normalised, dy)
    return pd.Series({"date": iso, "date_precision": precision})

entries[["date", "date_precision"]] = entries.apply(_standardise, axis=1)
entries["year_inferred"] = (
    entries["date"].notna() & ~entries["date_raw"].fillna("").str.contains(r"\b\d{4}\b")
)
entries["year"] = [
    int(d[:4]) if isinstance(d, str) and d[:4].isdigit() else (None if pd.isna(dy) else int(dy))
    for d, dy in zip(entries["date"], entries["default_year"])
]
entries["entry_text"] = entries["entry_text"].fillna("")

print(entries["date_precision"].value_counts().to_string())

date_precision
day      3169
month      50
year       47


## 3. Name folding and hand-edited resources

`fold` is the comparison form of a name, shared with the matching pipeline: lower case, accents removed, apostrophes and full stops removed, hyphens to spaces, Saint/Sainte to St/Ste.

The **lexicon** lists names whose type is known in advance (countries, provinces, historical regions, well-known ambiguous names). A name can have several rows with different year ranges; `ambiguous = 1` marks a default that context cues should override (*Quebec* is the city before 1867 and usually the province after). The lexicon also catches names the model misses (section 5.1).

The **not-places list** holds words that are never places in this corpus.

In [8]:
def fold(s) -> str:
    """Normalise a place string for comparison: accents, case, punctuation, Saint/St."""
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c)).lower()
    s = s.replace("'", "").replace("’", "").replace("‘", "").replace(".", "")
    s = re.sub(r"[-‐‑–—/]", " ", s)
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\bsaint\b", "st", s)
    s = re.sub(r"\bsainte\b", "ste", s)
    return " ".join(s.split())

def _to_int(x):
    try:
        return int(float(x))
    except (TypeError, ValueError):
        return None

lexicon_df = pd.read_csv(LEXICON_CSV, dtype=str, keep_default_na=False, encoding="utf-8")
LEXICON = defaultdict(list)      # folded name -> [(place_type, year_from, year_to, ambiguous)]
for r in lexicon_df.itertuples():
    LEXICON[fold(r.name)].append((r.place_type, _to_int(r.year_from), _to_int(r.year_to), r.ambiguous.strip() in ("1", "true", "True")))

not_places_df = pd.read_csv(NOT_PLACES_CSV, dtype=str, keep_default_na=False, encoding="utf-8")
NOT_PLACES = {fold(t) for t in not_places_df["term"]}

def lexicon_type(key, year):
    """(place_type, ambiguous) from the lexicon for a folded name and year, or None."""
    rows = [r for r in LEXICON.get(key, ())
            if year is None or ((r[1] is None or year >= r[1]) and (r[2] is None or year <= r[2]))]
    if not rows:
        return None
    return rows[0][0], rows[0][3] or len({r[0] for r in rows}) > 1

# Regular expression over lexicon surface forms, longest first, case-sensitive.
# Straight and curly apostrophes are treated as equal.
_lex_names = sorted(set(lexicon_df["name"]), key=len, reverse=True)
LEXICON_RE = re.compile(
    r"(?<![\w'’])(?:" + "|".join(re.escape(n).replace("'", "['’]") for n in _lex_names) + r")(?![\w'’])")

print(f"Lexicon: {len(lexicon_df)} rows, {len(LEXICON)} names | not-places: {len(NOT_PLACES)} terms")

Lexicon: 107 rows, 99 names | not-places: 46 terms


## 4. Named entity recognition

Runs the spaCy model over every entry and stores, per entry, every entity span (all labels, not only places, because person labels feed the person filter) and the sentence boundaries. Results are cached per book in `NER_CACHE_DIR/<model>/`, keyed by a hash of the entry text, so the filters and type rules below can be changed and re-run in seconds. A changed entry text or a different model is re-processed automatically.

> **[DESIGN NOTE — delete before publishing]**
> **Why `en_core_web_trf` and not `en_core_web_sm`.** The first prototype used the small model; most of its false places were character names ("Papa", "Armand", "Kateri") tagged `GPE` or `ORG`. The transformer model uses much more context and confuses people with places far less often. The cost is speed, which the cache absorbs.
>
> **Why `ORG` is no longer a place label.** About half of the prototype's mentions were `ORG`, and most of those were people, salutations ("Dear Papa and Mama") or institutions. `ORG` spans are now kept only when the same name is tagged as a place more often than not in that book, or when it is in the lexicon (`PROMOTE_ORG`).
>
> **Why the cache stores all labels.** The person filter (section 5.3) counts how often each name is tagged `PERSON` across a book. Storing everything means no second model pass is needed.

In [9]:
def model_slug():
    """Cache folder name for the configured model; a fine-tuned model path includes its build time."""
    p = Path(str(SPACY_MODEL))
    if p.exists():
        meta = p / "meta.json"
        return f"{p.name}_{int(meta.stat().st_mtime) if meta.exists() else 0}"
    return re.sub(r"\W+", "_", str(SPACY_MODEL))

def text_key(text):
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:16]

_nlp = None
def get_nlp():
    global _nlp
    if _nlp is None:
        _nlp = spacy.load(SPACY_MODEL, exclude=["lemmatizer"])   # the parser supplies sentence boundaries
        print("Loaded", _nlp.meta.get("name"), _nlp.meta.get("version"), "| pipeline:", _nlp.pipe_names)
    return _nlp

def _sentences(doc):
    try:
        return [[s.start_char, s.end_char] for s in doc.sents]
    except ValueError:                       # model without parser or sentencizer
        return [[0, len(doc.text)]]

MODEL_NAME = model_slug()
cache_dir = NER_CACHE_DIR / MODEL_NAME
cache_dir.mkdir(parents=True, exist_ok=True)

NER = {}                                     # entries index -> {"ents": [[start, end, label]], "sents": [[start, end]]}
for book, g in entries.groupby("book", sort=True):
    path = cache_dir / f"{book}.jsonl"
    cached = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            rec = json.loads(line)
            cached[rec["key"]] = rec
    keys = [text_key(t) for t in g["entry_text"]]
    todo = list(dict.fromkeys((k, t) for k, t in zip(keys, g["entry_text"]) if k not in cached))
    if todo:
        nlp = get_nlp()
        with path.open("a", encoding="utf-8") as f:
            docs = nlp.pipe([t for _, t in todo], batch_size=NER_BATCH_SIZE)
            for (k, _), doc in tqdm(zip(todo, docs), total=len(todo), desc=f"NER {book}"):
                rec = {"key": k, "ents": [[e.start_char, e.end_char, e.label_] for e in doc.ents],
                       "sents": _sentences(doc)}
                f.write(json.dumps(rec) + "\n")
                cached[k] = rec
    for i, k in zip(g.index, keys):
        NER[i] = cached[k]

label_counts = Counter(lab for r in NER.values() for _, _, lab in r["ents"])
print(f"Model {MODEL_NAME}: {sum(label_counts.values())} entities in {len(NER)} entries")
print(dict(label_counts.most_common()))

Model en_core_web_trf: 49127 entities in 3266 entries
{'PERSON': 28338, 'DATE': 5907, 'TIME': 2874, 'GPE': 2785, 'CARDINAL': 2492, 'NORP': 1830, 'FAC': 864, 'ORG': 850, 'ORDINAL': 751, 'LOC': 703, 'WORK_OF_ART': 663, 'PRODUCT': 279, 'EVENT': 235, 'LANGUAGE': 208, 'QUANTITY': 166, 'MONEY': 142, 'LAW': 31, 'PERCENT': 9}


## 5. Candidate mentions

### 5.1 Spans

A candidate is a model span whose label has role `place` or `org` in `MODEL_LABELS`, plus any lexicon name the model missed. A lexicon match replaces model spans it contains ("Red River Settlement" over "Red River"), and is skipped if it overlaps a person or another entity the model found ("Hudson's Bay" inside "the Hudson's Bay Company").

### 5.2 Cleaning

Whitespace is collapsed, curly apostrophes straightened, a leading article and surrounding punctuation removed. A trailing possessive `'s` is removed ("Canada's" → "Canada") except in names that end in it, such as "St. John's". The raw span is kept in `place_raw`.

In [10]:
_LEADING_ARTICLE = re.compile(r"^(?:the|a|an)\s+", re.I)
_KEEP_POSSESSIVE = re.compile(r"\bJohn's$")

def clean_mention(text: str) -> str:
    t = " ".join(text.split()).replace("’", "'").replace("‘", "'")
    t = _LEADING_ARTICLE.sub("", t)
    t = t.strip(" ,.;:!?\"'“”()[]")
    if t.endswith("'s") and fold(t) not in LEXICON and not _KEEP_POSSESSIVE.search(t):
        t = t[:-2]
    return t

def role_of(label):
    return MODEL_LABELS.get(label, ("other", None))

def candidate_spans(text, ents):
    cands = [dict(start=s, end=e, label=lab, role=role_of(lab)[0], model_type=role_of(lab)[1], source="ner")
             for s, e, lab in ents if role_of(lab)[0] in ("place", "org")]
    for m in LEXICON_RE.finditer(text):
        s, e = m.span()
        if any(c["start"] <= s and c["end"] >= e for c in cands):
            continue                                        # already covered by an equal or longer span
        if any(es < e and ee > s and role_of(lab)[0] not in ("place", "org") for es, ee, lab in ents):
            continue                                        # overlaps a person or other entity
        inside = [c for c in cands if c["start"] >= s and c["end"] <= e]
        if any(c["start"] < e and c["end"] > s for c in cands if c not in inside):
            continue                                        # partial overlap: keep the model's span
        for c in inside:
            cands.remove(c)
        cands.append(dict(start=s, end=e, label=inside[0]["label"] if inside else "", role="place",
                          model_type=inside[0]["model_type"] if inside else None,
                          source="ner+lexicon" if inside else "lexicon"))
    return sorted(cands, key=lambda c: c["start"])

### 5.3 Filters

Each candidate is checked in this order; the first failing check sets `filter_reason` and `keep = False`. Filtered mentions stay in the output.

| Reason | Rule |
|---|---|
| `too_short` | Fewer than three characters after folding (unless in the lexicon, e.g. "B.C."), or no letters |
| `not_capitalised` | Does not start with a capital letter ("n't", "the river") |
| `not_place_list` | In `not_places.csv` |
| `salutation` | Directly after "Dear", or starts with "Dear" ("Dear Papa and Mama") |
| `title_or_kinship` | Directly after a word in `PERSON_TITLES` ("Uncle Liam", "Mrs. Bell"), or starts with one (unless in the lexicon, e.g. "Father Point", or the title is in `TITLES_ALSO_IN_PLACE_NAMES`, e.g. "Prince Albert") |
| `mostly_person_in_book` | Across the whole book the name has at least as many person votes as place votes (unless in the lexicon) |
| `org_not_promoted` | An `ORG` span whose name is not mostly tagged as a place in the book and contains no generic place word ("Hotel Vancouver" and "Montreal General Hospital" are kept, "Parliament" is not), unless in the lexicon |

**Votes** are counted per book over folded names. A place vote is a span with role `place`. A person vote is a `PERSON` span equal to the name *or containing it as one of its words* ("Kirsty" in "Kirsty MacDonald"), or the name directly after a title anywhere in the book ("Madame Laurent").

> **[DESIGN NOTE — delete before publishing]**
> **Why votes across the book rather than per mention.** A diary repeats its characters' names hundreds of times. Even an accurate model occasionally tags a character as a place in an unusual sentence, and the prototype then mapped that character to a namesake in British Columbia or Romania. Counting all of a name's labels across the book turns many weak decisions into one strong one. Cost: a real place that shares a main character's name is filtered in that book; the lexicon is the escape hatch.
>
> **Why ties count as people.** In this corpus a capitalised name that the model is unsure about is more often a character than a place, and a wrongly kept person does more damage downstream (a false point on the map) than a wrongly dropped place (a missing point that the review table shows).

In [11]:
_TITLES = "|".join(sorted((re.escape(t) for t in PERSON_TITLES), key=len, reverse=True))
TITLE_BEFORE = re.compile(r"(?<![\w])(?:" + _TITLES + r")\.?\s+$")
_TITLES_START = "|".join(sorted((re.escape(t) for t in PERSON_TITLES if t not in TITLES_ALSO_IN_PLACE_NAMES),
                               key=len, reverse=True))
TITLE_START  = re.compile(r"^(?:" + _TITLES_START + r")\.?\s+\S")
TITLE_FOLLOWED = re.compile(r"(?<![\w])(?:" + _TITLES + r")\.?\s+([A-ZÀ-Ý][\w'’-]+)")
SALUTATION_BEFORE = re.compile(r"\bdear\s+$", re.I)

def book_votes(book_index):
    """{folded name: Counter(role -> count)} for one book."""
    votes = defaultdict(Counter)
    for i in book_index:
        text, rec = entries.at[i, "entry_text"], NER[i]
        for s, e, lab in rec["ents"]:
            role = role_of(lab)[0]
            name = clean_mention(text[s:e])
            key = fold(name)
            if role == "place":
                votes[key]["place"] += 1
            elif role == "org":
                votes[key]["org"] += 1
            elif role == "person":
                votes[key]["person"] += 1
                words = [fold(w) for w in name.split()]
                for w in (set(words) if len(words) > 1 else ()):
                    votes[w]["person"] += 1
        for m in TITLE_FOLLOWED.finditer(text):
            votes[fold(m.group(1))]["person"] += 1
    return votes

def filter_reason(c, text, votes):
    place, key = c["place"], c["place_key"]
    if (len(key) < 3 and key not in LEXICON) or not any(ch.isalpha() for ch in place):
        return "too_short"
    if not place[:1].isupper():
        return "not_capitalised"
    if key in NOT_PLACES:
        return "not_place_list"
    before = text[max(0, c["start"] - 40):c["start"]]
    if SALUTATION_BEFORE.search(before) or re.match(r"dear\b", place, re.I):
        return "salutation"
    in_lexicon = key in LEXICON
    if TITLE_BEFORE.search(before) or (TITLE_START.match(place) and not in_lexicon):
        return "title_or_kinship"
    v = votes.get(key, Counter())
    if not in_lexicon and v["person"] > 0 and v["person"] >= v["place"]:
        return "mostly_person_in_book"
    if c["role"] == "org" and not in_lexicon and not (
            PROMOTE_ORG and (v["place"] > v["person"] + v["org"] or head_type(place))):
        return "org_not_promoted"
    return None

## 6. Place type

The first rule that applies sets `place_type`, `type_confidence` and `type_evidence`:

| Order | Rule | Confidence |
|---|---|---|
| 1 | The model label implies a type (fine-tuned model only) | `model` |
| 2 | The lexicon gives one unambiguous type for the name and the entry's year ("Prince Edward Island" is a region, not an island) | `high` |
| 3 | A generic word is part of the name: last word in `TYPE_HEADS_LAST` ("Red River"), first word in `TYPE_HEADS_FIRST` ("Lake Ontario") | `high` |
| 4 | The word directly after the name is in `TYPE_HEADS_LAST` ("Quebec City" when only "Quebec" was tagged) | `high` |
| 5 | A phrase in `TYPE_CUES_BEFORE` directly precedes the name ("the city of Quebec", "aboard the Titanic") | `medium` |
| 6 | The lexicon gives a default for an ambiguous name | `medium` |
| 7 | Fallback by model label (`LABEL_FALLBACK_TYPE`) | `low` |

Context cues (rules 3–5) come before ambiguous lexicon defaults, so *"the province of Quebec"* in 1850 is a region even though the lexicon's pre-1867 default for a bare "Quebec" is the city.

> **[DESIGN NOTE — delete before publishing]**
> **Why rules, and why this order.** Without labelled data, the words around a name are the most reliable signal: diarists write "the town of", "aboard the", "Lake". The rules are deliberately few and visible. Every output row records which rule fired (`type_evidence`), so the review tables show which rules over- or under-fire. When gold annotations exist, section 8 measures type accuracy per rule, and a fine-tuned model can take over through rule 1.
>
> **Why ambiguous lexicon names get `medium` confidence.** The matching pipeline uses confidence to decide how strictly a gazetteer feature must agree with the type. A `medium` type still steers the match (city vs province) but the matcher accepts a different feature class if no compatible feature exists.

In [12]:
def _cue_regex(phrases):
    alts = "|".join(sorted((re.escape(p) for p in phrases), key=len, reverse=True))
    return re.compile(r"(?<![\w])(?:" + alts + r")\s+(?:the\s+)?$", re.I)

CUES_BEFORE = {t: _cue_regex(p) for t, p in TYPE_CUES_BEFORE.items()}
HEADS_LAST  = {t: set(v) for t, v in TYPE_HEADS_LAST.items()}
HEADS_FIRST = {t: set(v) - WEAK_FIRST_WORDS for t, v in TYPE_HEADS_FIRST.items()}
_NEXT_WORD  = re.compile(r"^\s+([A-Za-zÀ-ÿ]+)")

def head_type(place):
    """Type implied by a generic word that is part of a multi-word name, or None."""
    words = [fold(w) for w in place.split()]
    if len(words) > 1:
        for t, heads in HEADS_LAST.items():
            if words[-1] in heads:
                return t, words[-1]
        for t, heads in HEADS_FIRST.items():
            if words[0] in heads:
                return t, words[0]
    return None

def suggest_type(c, text, year):
    """(place_type, confidence, evidence) for one candidate."""
    if c["model_type"]:
        return c["model_type"], "model", f"model label {c['label']}"
    lex = lexicon_type(c["place_key"], year)
    if lex and not lex[1]:
        return lex[0], "high", "lexicon"
    head = head_type(c["place"])
    if head:
        return head[0], "high", f"name contains '{head[1]}'"
    nxt = _NEXT_WORD.match(text[c["end"]:c["end"] + 30])
    if nxt and nxt.group(1)[0].isupper():
        w = fold(nxt.group(1))
        for t, heads in HEADS_LAST.items():
            if w in heads:
                return t, "high", f"followed by '{nxt.group(1)}'"
    before = text[max(0, c["start"] - 40):c["start"]]
    for t, rx in CUES_BEFORE.items():
        m = rx.search(before)
        if m:
            return t, "medium", f"cue '{m.group(0).strip()}'"
    if lex:
        return lex[0], "medium", "lexicon (ambiguous)"
    fallback = LABEL_FALLBACK_TYPE.get(c["label"])
    if fallback:
        return fallback, "low", f"label {c['label']}"
    return "unknown", "low", "none"

## 7. Assemble and write outputs

In [13]:
rows = []
for book, g in tqdm(entries.groupby("book", sort=True), desc="Candidates"):
    votes = book_votes(g.index)
    for i in g.index:
        e = entries.loc[i]
        text, rec = e["entry_text"], NER[i]
        sent_starts = [s for s, _ in rec["sents"]]
        for c in candidate_spans(text, rec["ents"]):
            c["place"] = clean_mention(text[c["start"]:c["end"]])
            c["place_key"] = fold(c["place"])
            reason = filter_reason(c, text, votes)
            ptype, conf, evidence = suggest_type(c, text, e["year"])
            si = max(0, bisect.bisect_right(sent_starts, c["start"]) - 1)
            ss, se = rec["sents"][si] if rec["sents"] else (0, len(text))
            v = votes.get(c["place_key"], Counter())
            rows.append({
                "mention_id": f"{book}:{e['entry_id']}:{c['start']}",
                "book": book, "chapter_id": e["chapter_id"], "entry_id": e["entry_id"],
                "date": e["date"], "date_precision": e["date_precision"], "year_inferred": e["year_inferred"],
                "date_raw": e["date_raw"], "year": e["year"],
                "place": c["place"], "place_raw": text[c["start"]:c["end"]], "place_key": c["place_key"],
                "char_start": c["start"], "char_end": c["end"],
                "sentence": " ".join(text[ss:se].split()),
                "ner_label": c["label"], "source": c["source"],
                "votes": f"place:{v['place']}|person:{v['person']}|org:{v['org']}",
                "keep": reason is None, "filter_reason": reason or "",
                "place_type": ptype, "type_confidence": conf, "type_evidence": evidence,
                "model": MODEL_NAME,
            })

OUTPUT_COLUMNS = ["mention_id", "book", "chapter_id", "entry_id", "date", "date_precision", "year_inferred",
                  "date_raw", "year", "place", "place_raw", "place_key", "char_start", "char_end", "sentence",
                  "ner_label", "source", "votes", "keep", "filter_reason", "place_type", "type_confidence",
                  "type_evidence", "model"]
mentions = pd.DataFrame(rows, columns=OUTPUT_COLUMNS)
mentions["year"] = mentions["year"].astype("Int64")
assert mentions["mention_id"].is_unique

kept = mentions[mentions["keep"]]
names = (mentions.groupby(["book", "place_key"])
         .agg(place=("place", lambda s: s.value_counts().index[0]),
              n_candidates=("place", "size"), n_kept=("keep", "sum"),
              filter_reasons=("filter_reason", lambda s: "|".join(sorted(set(s) - {""}))),
              votes=("votes", "first"),
              types=("place_type", lambda s: "|".join(f"{k}:{n}" for k, n in s.value_counts().items())),
              labels=("ner_label", lambda s: ",".join(sorted(set(s) - {""}))))
         .reset_index()
         .sort_values(["book", "n_candidates"], ascending=[True, False]))

mentions.to_csv(EXTRACTED_CSV, index=False, encoding="utf-8")
names.to_csv(NAMES_SUMMARY_CSV, index=False, encoding="utf-8")
print(f"{len(mentions)} candidates, {len(kept)} kept ({len(kept) / max(1, len(mentions)):.0%})")
print(f"Wrote -> {EXTRACTED_CSV}")
print(f"Wrote -> {NAMES_SUMMARY_CSV}")

Candidates:   0%|          | 0/20 [00:00<?, ?it/s]

5232 candidates, 4384 kept (84%)
Wrote -> /Users/sophiebird/Downloads/Dear Canada/Pipelines/Place Mentions/intermediate/place_mentions_extracted.csv
Wrote -> /Users/sophiebird/Downloads/Dear Canada/Pipelines/Place Mentions/intermediate/place_names_by_book.csv


## 7.1 Review

These tables are the main tool for tuning the filters, the type rules and the lexicon.

In [14]:
print("Filter reasons")
print(mentions["filter_reason"].replace("", "(kept)").value_counts().to_string())
print("\nKept mentions by type and confidence")
print(pd.crosstab(kept["place_type"], kept["type_confidence"], margins=True).to_string())
print("\nKept mentions per book")
print(kept.groupby("book").size().to_string())

Filter reasons
filter_reason
(kept)                   4384
org_not_promoted          576
mostly_person_in_book     145
salutation                 46
not_capitalised            42
not_place_list             15
title_or_kinship           12
too_short                  12

Kept mentions by type and confidence
type_confidence  high   low  medium   All
place_type                               
country           551     0     335   886
facility          510   384       0   894
landform           88     0       0    88
region            334     0      70   404
settlement         61  1265     234  1560
unknown             0   282       0   282
vessel             31     0       0    31
water             230     0       9   239
All              1805  1931     648  4384

Kept mentions per book
book
AToBD      163
AinUTL     163
BFfH       121
BUoL       225
CofOO      322
DRtoF      225
DofTT      137
DofmyC     163
EftW       240
FitS       215
NSH        265
NbutC      152
OA         104
OatmyD 

In [15]:
print("Most frequent kept names (check for people that slipped through)")
(kept.groupby("place_key")
     .agg(place=("place", lambda s: s.value_counts().index[0]), n=("place", "size"),
          books=("book", lambda s: ",".join(sorted(set(s)))),
          types=("place_type", lambda s: "|".join(f"{k}:{n}" for k, n in s.value_counts().items())))
     .sort_values("n", ascending=False).head(40))

Most frequent kept names (check for people that slipped through)


,place,n,books,types
place_key,,,,
canada,Canada,337,"BFfH,BUoL,CofOO,DRtoF,DofTT,DofmyC,EftW,NSH,Nb...",country:335|region:2
england,England,177,"BFfH,BUoL,CofOO,DRtoF,DofmyC,EftW,FitS,NSH,Nbu...",country:177
montreal,Montreal,123,"AinUTL,BFfH,BUoL,CofOO,DofmyC,FitS,NSH,NbutC,P...",settlement:118|unknown:4|facility:1
quebec,Québec,92,"AinUTL,BUoL,CofOO,DofmyC,NbutC,PintPL,SofS,TA,...",settlement:78|region:14
france,France,88,"AinUTL,BFfH,BUoL,DofmyC,EftW,NSH,PintPL,SofS,T...",country:88
toronto,Toronto,84,"AToBD,BFfH,BUoL,CofOO,DRtoF,DofTT,EftW,OA,Oatm...",settlement:84
ottawa,Ottawa,61,"BUoL,CofOO,DofTT,OA,TA",settlement:59|water:2
halifax,Halifax,54,"BFfH,EftW,NSH,TFN",settlement:52|unknown:2
vancouver,Vancouver,46,"NSH,PasWide,TA",settlement:46


In [16]:
print("Most frequent filtered names (check for places that were dropped)")
(mentions[~mentions["keep"]].groupby(["place_key", "filter_reason"])
     .agg(place=("place", lambda s: s.value_counts().index[0]), n=("place", "size"),
          books=("book", lambda s: ",".join(sorted(set(s)))))
     .reset_index().sort_values("n", ascending=False).head(40))

Most frequent filtered names (check for places that were dropped)


,place_key,filter_reason,place,n,books
52,canada,salutation,Canada,43,"CofOO,OatmyD,PasWide,TFN,WofW"
250,north west company,org_not_promoted,North West Company,23,"BUoL,FitS"
86,confederation,org_not_promoted,Confederation,17,CofOO
204,le bon dieu,not_capitalised,le bon Dieu,17,"AinUTL,DofmyC"
178,hudsons bay company,org_not_promoted,Hudson's Bay Company,17,"BUoL,CofOO,FitS"
81,commission,org_not_promoted,Commission,15,TA
149,guide,org_not_promoted,Guide,14,TA
284,red cross,org_not_promoted,Red Cross,13,"BFfH,EftW,NSH,PintPL,TA"
150,guides,org_not_promoted,Guides,12,TA
84,company,org_not_promoted,Company,11,FitS


In [17]:
print("Names with more than one type in the same book (e.g. Quebec city vs province)")
multi = kept.groupby(["book", "place_key"])["place_type"].nunique()
(kept.set_index(["book", "place_key"]).loc[multi[multi > 1].index]
     .reset_index()[["book", "place", "place_type", "type_evidence", "sentence"]].head(30))

Names with more than one type in the same book (e.g. Quebec city vs province)


,book,place,place_type,type_evidence,sentence
0,AToBD,St. Anne,settlement,label GPE,Never have I felt so glad to see a place as th...
1,AToBD,St. Anne,facility,label FAC,He is to stay at St. Anne’s until he recovers.
2,AToBD,St. Anne,facility,label FAC,"I spent a fair amount of time studying him, as..."
3,AinUTL,Le Cadeau,unknown,label ORG,Le Cadeau has twenty rooms!
4,AinUTL,Le Cadeau,facility,label FAC,Monsieur Deschamps has travelled on ahead to t...
5,AinUTL,Le Cadeau,facility,label FAC,"Madame turned an unpleasant shade of purple, o..."
6,AinUTL,Le Cadeau,facility,label FAC,"Although they could not paddle to it, Madame L..."
7,AinUTL,Le Cadeau,facility,label FAC,"Since I left Le Cadeau , so many changes have ..."
8,AinUTL,Le Cadeau,facility,label FAC,"How she enjoyed this day back at Le Cadeau , t..."
9,AinUTL,Montréal,settlement,label GPE,"He lives with his family in Montréal, across t..."


## 8. Annotation, evaluation and fine-tuning

The path from rules to a trained model:

1. **Export a sample** (8.1). Entries are written to `annotation/to_annotate/` as JSONL, pre-annotated with the current kept predictions as spans labelled by type (`SETTLEMENT`, `REGION`, `COUNTRY`, `WATER`, `LANDFORM`, `FACILITY`, `VESSEL`, `PLACE` for unknown). The format is the one used by Prodigy and spaCy (`{"text", "spans": [{"start", "end", "label"}], "meta"}`); Label Studio and doccano import it with a small conversion.
2. **Correct by hand** and save the corrected files to `annotation/gold/`. Mark every place, fix the span edges and the type, delete false places. Anything not marked is treated as not a place. Story locations with a proper name ("Halls' Farm") can be marked with their type like any other place.
3. **Evaluate** (8.2). Compares the current predictions with the gold entries: span precision and recall, type accuracy, and which filter wrongly removed gold places. Run it before and after every change to the rules.
4. **Train** (8.3). Builds `train.spacy` and `dev.spacy` from the gold entries and prints the commands to fine-tune a model. Then set `SPACY_MODEL` to the trained model's folder; its type labels are used directly through `MODEL_LABELS` (type rule 1). The filters and the evaluation work unchanged.

### 8.1 Export a sample for annotation

In [18]:
ANNOTATION_SAMPLE_PER_BOOK = 5           # entries per book
ANNOTATION_SEED = 0
TYPE_TO_LABEL = {"settlement": "SETTLEMENT", "region": "REGION", "country": "COUNTRY", "water": "WATER",
                 "landform": "LANDFORM", "facility": "FACILITY", "vessel": "VESSEL", "unknown": "PLACE"}

def read_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]

def gold_records():
    recs = []
    for f in sorted(GOLD_DIR.glob("*.jsonl")):
        recs += [r for r in read_jsonl(f) if r.get("answer", "accept") == "accept"]
    return recs

def export_annotation_sample(n_per_book=ANNOTATION_SAMPLE_PER_BOOK, seed=ANNOTATION_SEED):
    done = {(r["meta"]["book"], r["meta"]["entry_id"]) for r in gold_records()}
    rng = random.Random(seed)
    out = []
    for book, g in entries.groupby("book", sort=True):
        pool = [i for i in g.index if (book, g.at[i, "entry_id"]) not in done and g.at[i, "entry_text"].strip()]
        for i in sorted(rng.sample(pool, min(n_per_book, len(pool)))):
            e = entries.loc[i]
            m = kept[(kept["book"] == book) & (kept["entry_id"] == e["entry_id"])]
            out.append({"text": e["entry_text"],
                        "spans": [{"start": int(s), "end": int(t), "label": TYPE_TO_LABEL[p]}
                                  for s, t, p in zip(m["char_start"], m["char_end"], m["place_type"])],
                        "meta": {"book": book, "entry_id": e["entry_id"], "chapter_id": e["chapter_id"],
                                 "date": e["date"], "model": MODEL_NAME}})
    path = TO_ANNOTATE_DIR / f"sample_{n_per_book}per_book_seed{seed}.jsonl"
    path.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in out) + "\n", encoding="utf-8")
    print(f"Wrote {len(out)} entries -> {path}")

export_annotation_sample()

Wrote 100 entries -> /Users/sophiebird/Downloads/Dear Canada/Pipelines/Place Mentions/annotation/to_annotate/sample_5per_book_seed0.jsonl


### 8.2 Evaluate against gold annotations

- **Span**: a prediction is correct if its offsets equal a gold span (`exact`), or overlap one (`overlap`, which forgives "the Ottawa River" vs "Ottawa River").
- **Type accuracy**: among overlapping pairs, the share where the predicted type equals the gold label.
- **Filter losses**: gold places that were found but removed by a filter, by reason.

Gold entries are matched to the corpus by `(book, entry_id)`; an entry whose text has changed since annotation is skipped with a warning.

In [19]:
LABEL_TO_TYPE = {v: k for k, v in TYPE_TO_LABEL.items()}

def evaluate(gold):
    by_entry = {(b, e): g for (b, e), g in mentions.groupby(["book", "entry_id"])}
    text_of = {(b, e): t for b, e, t in zip(entries["book"], entries["entry_id"], entries["entry_text"])}
    tp_exact = tp_overlap = n_pred = n_gold = 0
    type_pairs, filter_losses, per_book, errors = [], Counter(), defaultdict(Counter), []
    for r in gold:
        key = (r["meta"]["book"], r["meta"]["entry_id"])
        if text_of.get(key) != r["text"]:
            print("WARNING: entry text changed or missing, skipped:", key)
            continue
        m = by_entry.get(key, mentions.iloc[0:0])
        pred = [(s, e, t) for s, e, t, k in zip(m["char_start"], m["char_end"], m["place_type"], m["keep"]) if k]
        dropped = [(s, e, why) for s, e, why, k in zip(m["char_start"], m["char_end"], m["filter_reason"], m["keep"]) if not k]
        gold_spans = [(s["start"], s["end"], LABEL_TO_TYPE.get(s["label"], "unknown")) for s in r.get("spans", [])]
        n_pred += len(pred); n_gold += len(gold_spans)
        per_book[key[0]].update(pred=len(pred), gold=len(gold_spans))
        for gs, ge, gt in gold_spans:
            hit = [p for p in pred if p[0] < ge and p[1] > gs]
            if any(p[0] == gs and p[1] == ge for p in pred):
                tp_exact += 1
            if hit:
                tp_overlap += 1
                per_book[key[0]]["tp"] += 1
                type_pairs.append((gt, hit[0][2]))
            else:
                lost = [why for s, e, why in dropped if s < ge and e > gs]
                if lost:
                    filter_losses[lost[0]] += 1
                errors.append(dict(book=key[0], entry_id=key[1], kind="missed", text=r["text"][gs:ge],
                                   gold_type=gt, filtered_by=lost[0] if lost else ""))
        for ps, pe, pt in pred:
            if not any(gs < pe and ge > ps for gs, ge, _ in gold_spans):
                errors.append(dict(book=key[0], entry_id=key[1], kind="false_place", text=r["text"][ps:pe],
                                   gold_type="", filtered_by=""))
    f1 = lambda p, r: 0 if p + r == 0 else 2 * p * r / (p + r)
    p_ex, r_ex = tp_exact / max(1, n_pred), tp_exact / max(1, n_gold)
    p_ov, r_ov = tp_overlap / max(1, n_pred), tp_overlap / max(1, n_gold)
    print(f"Gold spans {n_gold}, predicted {n_pred}")
    print(f"exact   P={p_ex:.2f} R={r_ex:.2f} F1={f1(p_ex, r_ex):.2f}")
    print(f"overlap P={p_ov:.2f} R={r_ov:.2f} F1={f1(p_ov, r_ov):.2f}")
    if type_pairs:
        print(f"type accuracy on overlapping pairs: {sum(g == p for g, p in type_pairs) / len(type_pairs):.2f}")
        print(pd.crosstab(pd.Series([g for g, _ in type_pairs], name="gold"),
                          pd.Series([p for _, p in type_pairs], name="predicted")).to_string())
    if filter_losses:
        print("gold places removed by filters:", dict(filter_losses))
    return pd.DataFrame(errors)

GOLD = gold_records()
if GOLD:
    eval_errors = evaluate(GOLD)
    display(eval_errors.head(40))
else:
    print(f"No gold annotations in {GOLD_DIR} yet; see 8.1.")

No gold annotations in /Users/sophiebird/Downloads/Dear Canada/Pipelines/Place Mentions/annotation/gold yet; see 8.1.


### 8.3 Build training data for fine-tuning

Writes `train.spacy` and `dev.spacy` (80/20 split by entry) to `annotation/spacy/`. Spans are aligned to tokens; spans that cannot be aligned are reported and skipped.

Suggested route, once a few hundred entries are annotated: keep the pre-trained transformer and train its NER component on the new labels.

```bash
cd "Pipelines/Place Mentions/annotation"
python -m spacy init config spacy/config.cfg --lang en --pipeline ner --optimize accuracy
# In config.cfg: source the transformer from en_core_web_trf, e.g.
#   [components.transformer]  source = "en_core_web_trf"
python -m spacy train spacy/config.cfg --output models --paths.train spacy/train.spacy --paths.dev spacy/dev.spacy
```

Then set `SPACY_MODEL = MODELS_DIR / "model-best"` and re-run the notebook. The fine-tuned labels map to types through `MODEL_LABELS`.

A fine-tuned model no longer produces `PERSON` spans unless they are annotated too. The person filter then relies on title cues only, so either annotate `PERSON` as well or keep running the pre-trained model alongside for votes.

In [20]:
TRAIN_SHARE = 0.8

def build_training_data(gold, seed=0):
    from spacy.tokens import DocBin
    nlp_blank = spacy.blank("en")
    recs = list(gold)
    random.Random(seed).shuffle(recs)
    cut = int(len(recs) * TRAIN_SHARE)
    SPACY_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for name, part in (("train", recs[:cut]), ("dev", recs[cut:])):
        db, skipped = DocBin(), 0
        for r in part:
            doc = nlp_blank.make_doc(r["text"])
            spans = []
            for s in r.get("spans", []):
                span = doc.char_span(s["start"], s["end"], label=s["label"], alignment_mode="contract")
                if span is None:
                    skipped += 1
                else:
                    spans.append(span)
            doc.ents = spacy.util.filter_spans(spans)
            db.add(doc)
        db.to_disk(SPACY_DATA_DIR / f"{name}.spacy")
        print(f"{name}: {len(part)} entries -> {SPACY_DATA_DIR / (name + '.spacy')}  ({skipped} spans not aligned)")

if GOLD:
    build_training_data(GOLD)
else:
    print("No gold annotations yet.")

No gold annotations yet.
